## 3. Module 2 — Graph-Based Analysis
**Initial task description (Module 2 perspective):**
Building upon our structural discoveries from Module 1, our second analysis phase shifts the focus from global geometric alignment to local topological structures. Evidence suggests that calculating classical distances across a sparse, high dimensional text space is inherently noisy, which is not ideal to capture the nuanced semantic boundaries. To address this limitation, we model the data as a graph, transforming textual similarities into a network topology.

We implemented two graph mining algorithms to identify these contextual boundaries. We selected Spectral Clustering and the Louvain method because they represent two fundamentally opposing topological paradigms: top-down global partitioning and bottom-up local agglomeration. While our overarching shift to a graph-based approach aims to structurally mitigate the spatial "gravity well" effect, the juxtaposition of these specific algorithms provides deeper behavioral insight. Specifically, it evaluates whether the human musical curation landscape is more accurately modeled as a globally partitioned space defined by distinct boundaries, or as a locally agglomerated space driven by internal cohesion.


### Preprocessing
The primary preprocessing step in this phase involves converting our global TF-IDF matrix into a local k-NN similarity graph. We chose this text-based graph conversion method over a behavior-based (such as a shared-track bipartite graph) for three methodological reasons: it preserves our core research question regarding the semantic value of playlist names, it ensures a controlled comparison against our baseline by keeping the input features constant, and it acts as a structural remedy to the "gravity well" effect observed part 1 by mathematically severing generic ties and isolating relevant local neighborhoods.

To do this, we calculate the Cosine Similarity using the values of the TF-IDF between each playlist. Thereafter we sparsify the values, converting all but the top 10 highest numbers for each row to 0. The result is a weighted adjacency matrix where each playlist is a node, and every non-zero number becomes an edge. We choose k to be 10 to ensure inclusivity while preventing massive node hubs, and we define a minimum threshold of 0.15 to ensure we don't force connections of low quality.

Crucially, this process provides new opportunities for structural analysis that traditional geometric clustering cannot offer. The new network representation allows us to model musical curation as interconnected communities, which establishes the ideal foundational topology for our graph mining methodologies, which identify natural boundaries based on network density rather than global spatial distance.

In [ ]:
from graph.knn.knn_graph import KNNGraph

In [ ]:
graph_builder = KNNGraph(k_neighbors=10, sim_threshold=0.15)
    
graph_config_name = f"k{graph_builder.k}_sim{graph_builder.sim_threshold}_N{len(unique_texts)}"

graph_save_dir = os.path.join("graph", "knn", "saved_graphs")
os.makedirs(graph_save_dir, exist_ok=True)

graph_save_path = os.path.join(graph_save_dir, f"knn_{graph_config_name}.pkl")

if os.path.exists(graph_save_path):
    print(f"\n[INFO] Loading previously built k-NN graph from {graph_save_path}...")
    with open(graph_save_path, "rb") as f:
        graph_builder.G = pickle.load(f)
else:
    print("\n[INFO] Building shared k-NN graph for graph-based clustering...")
    graph_builder.build_graph(tfidf_matrix, unique_texts)
    
    print(f"[INFO] Saving generated k-NN graph to {graph_save_path}...")
    with open(graph_save_path, "wb") as f:
        pickle.dump(graph_builder.G, f)

### Louvain Community Detection
The Louvain method is a heuristic bottom up approach designed to discover communities by Maximizing Modularity.
Modularity is a scalar value that quantifies the density of a random graph. The algorithm operates in a continuous 2
phase loop, phase 1 allowing nodes to iteratively jump to neighboring communities that yield the highest increase in
modularity, and phase 2 aggregating the communities into "super nodes". This continues until the global modularity can
no longer be improved. This allows the method to discover the optimal communities.
We specifically chose the Louvain method over Ravasz and link clustering as both run O($n^2$) complexity, as
opposed to Louvain’s O($m$).

#### Executing Louvain Community Detection - $k = 322$
We execute the Louvain algorithm using the python-louvain library, specifically invoking the best_partition() function on our interconnected network of playlists.

When invoking the algorithm, we pass our k-NN graph directly into the function without explicitly overriding the default hyperparameters. We designed our execution strategy around two default parameters:

- **weight='weight'**: Modularity works by evaluating edge densities. We explicitly stored our TF-IDF cosine similarities under the "weight" edge attribute during the k-NN graph construction and hence Louvain automatically detects and leverages this data. This  upgrades the algorithm from evaluating basic, binary network connections to evaluating the contextual equivalence between playlists.
- **resolution=1.0**: As we are basing our project on structure discovery, we define the default resolution, as this is where the modularity equation is balanced and where clusters are defined naturally based on density.

In [ ]:
# Initialize Louvain Clustering instance, and load the graph from the shared builder
from graph.knn.louvain_clustering import LouvainClustering
Louvain = LouvainClustering(graph=graph_builder.G, graph_config_name=graph_config_name, random_state=RANDOM_SEED)

In [ ]:
# Get the cluster_col name, which we use to accesss / save the data frame
louvain_col = getattr(Louvain, "cluster_col", None)
# Report output directory for Louvain
louvain_report_out = Louvain.report_dir

In [ ]:
# Check if the Louvain column already exists to avoid redundant computation
if louvain_col in df.columns:
    print(f"\n[SKIP] {Louvain.algo_name} already exists in column '{louvain_col}'.")

# If the column doesn't exist, we run the full pipeline for Louvain clustering
else:
    print(f"\n{'='*50}")
    print(f"Executing Pipeline for: {Louvain.algo_name}")
    print(f"{'='*50}")

    # Pipeline execution
    df, target_col = Louvain.run_pipeline(df, unique_texts, tfidf_matrix)
    # Report creation
    Louvain.create_report()

    # If the pipeline produced a new target column, we save the updated dataframe with the new labels
    if target_col and target_col in df.columns:
        from preprocessing.preprocessor import FULLY_PROCESSED_PARQUET
        print(f"[INFO] Saving updated labels to {FULLY_PROCESSED_PARQUET}...")
        df.to_parquet(FULLY_PROCESSED_PARQUET, index=False)

### Spectral Clustering
Spectral Clustering is a deterministic partitioning technique rooted in linear algebra that identifies optimal global cuts.
It uses the Laplacian matrix to find eigenvectors/eigenvalues and uses them to project the complex topology into a
lower dimensional spectral space. Dense communities mathematically pull apart, and K-means can be applied to find k
distinct partitions.
We specifically chose Spectral Clustering as opposed to similar alternatives due to its performance on sparse
k-NN graphs, in which libraries like scikit-learn allows complexity of $O(km)$ (where $m$ is the number of edges.)

#### Executing Spectral Clustering - $k = 322$
To execute the algorithm, we utilize the SpectralClustering implementation from the scikit-learn library. Spectral Clustering requires a mathematical representation of our interconnected network. We translate our networkx k-NN graph into a SciPy sparse adjacency matrix, explicitly ensuring that the algorithm inherits our TF-IDF cosine similarities by preserving the edge weight attributes.

We instantiate the model with the following hyperparameters:
- **n_clusters=???**: We match it with the final cluster value from Louvain, for a fair comparison.
- **affinity="precomputed"**: This parameter instructs the algorithm to bypass its internal similarity calculations and instead accept our spatial matrix. We chose this to ensure the model evaluates the network based on the TF-IDF cosine similarities we established during the k-NN graph construction.
- **eigen_solver="arpack"**: The ARPACK solver was explicitly chosen over standard solvers because it is mathematically much more stable when handling disjointed network topologies such as ours.
- **assign_labels="cluster_qr"**: We use QR decomposition for the final labeling step rather than the default k-means approach, as it provides a significantly faster and strictly deterministic result.

In [ ]:
# Initialize Spectral Clustering instance, and load the graph from the shared builder
from graph.knn.spectral_clustering import SpectralGraphClustering
Spectral_322 = SpectralGraphClustering(graph=graph_builder.G, n_clusters=322, graph_config_name=graph_config_name)

In [ ]:
# Get the cluster_col name, which we use to accesss / save the data frame
spectral_col_322 = getattr(Spectral_322, "cluster_col", None)
# Report output directory for Spectral
spectral_report_out_322 = Spectral_322.report_dir

In [ ]:
# Check if the Spectral column already exists to avoid redundant computation
if spectral_col_322 in df.columns:
    print(f"\n[SKIP] {Spectral_322.algo_name} already exists in column '{spectral_col_322}'.")

# If the column doesn't exist, we run the full pipeline for Spectral clustering
else:
    print(f"\n{'='*50}")
    print(f"Executing Pipeline for: {Spectral_322.algo_name}")
    print(f"{'='*50}")

    # Pipeline execution
    df, target_col = Spectral_322.run_pipeline(df, unique_texts, tfidf_matrix)
    # Report creation
    Spectral_322.create_report()

    # If the pipeline produced a new target column, we save the updated dataframe with the new labels
    if target_col and target_col in df.columns:
        from preprocessing.preprocessor import FULLY_PROCESSED_PARQUET
        print(f"[INFO] Saving updated labels to {FULLY_PROCESSED_PARQUET}...")
        df.to_parquet(FULLY_PROCESSED_PARQUET, index=False)

### Evaluation
#### Spectral Clustering Evaluation

In [ ]:
# Safety Net
if spectral_col_322 in df.columns:
    os.makedirs(spectral_report_out_322, exist_ok=True)
    spectral_report_path = os.path.join(spectral_report_out_322, f"evaluation_metrics_{spectral_col_322}.txt")
    # Skip Logic
    if os.path.exists(spectral_report_path):
        print(f"\n[SKIP] Evaluation for {Spectral_322.algo_name} already exists at '{spectral_report_path}'.")
    else:
        print(f"\n[INFO] Column '{spectral_col_322}' exists but no evaluation report found. Running eval for {Spectral_322.algo_name}...")
        eval(df=df,
             cluster_col=spectral_col_322,
             unique_texts=unique_texts,
             tfidf_matrix=tfidf_matrix,
             sample_frac=0.1,
             output_dir=spectral_report_out_322)

#### Louvain Community Detection Evaluation

In [ ]:
# Safety Net
if louvain_col in df.columns:
    os.makedirs(louvain_report_out, exist_ok=True)
    louvain_report_path = os.path.join(louvain_report_out, f"evaluation_metrics_{louvain_col}.txt")
    # Skip Logic
    if os.path.exists(louvain_report_path):
        print(f"\n[SKIP] Evaluation for {Louvain.algo_name} already exists at '{louvain_report_path}'.")
    else:
        print(f"\n[INFO] Column '{louvain_col}' exists but no evaluation report found. Running eval for {Louvain.algo_name}...")
        eval(df=df,
             cluster_col=louvain_col,
             unique_texts=unique_texts,
             tfidf_matrix=tfidf_matrix,
             sample_frac=0.1,
             output_dir=louvain_report_out)

##### Plotting $F_{0.1}$-scores

In [ ]:
# Plotting the results side by side for comparison
spectral_322_path_fscore = os.path.join(spectral_report_out_322, "f01_comparison.png")
louvain_path_fscore = os.path.join(louvain_report_out, "f01_comparison.png")

spectral_322_path_dist = os.path.join(spectral_report_out_322, "cluster_distribution.png")
louvain_path_dist = os.path.join(louvain_report_out, "cluster_distribution.png")

# Plotting them side by side
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes[0, 0].imshow(plt.imread(spectral_322_path_fscore))
axes[0, 0].set_title(f"{Spectral_322.algo_name} F0.1-Score Comparison")
axes[0, 0].axis('off')
axes[0, 1].imshow(plt.imread(louvain_path_fscore))
axes[0, 1].set_title(f"{Louvain.algo_name} F0.1-Score Comparison")
axes[1, 0].imshow(plt.imread(spectral_322_path_dist))
axes[1, 0].set_title(f"{Spectral_322.algo_name} Cluster Distribution")
axes[1, 0].axis('off')
axes[1, 1].imshow(plt.imread(louvain_path_dist))
axes[1, 1].set_title(f"{Louvain.algo_name} Cluster Distribution")
axes[1, 1].axis('off')
plt.tight_layout()
plt.show()

##### Evaluating $F_{0.1}$-scores 
Both Spectral and Louvain achieve great performance in the Top-1 bracket (around 0.9). Meaning that if we only look at the absolute closests neighbors, both algorithms successfully group highly relevant playlists together. 

As we expand the recommendation pool, Spectral's Top-5 and Top-10 scores stay relatively high (about 0.85 and 0.80), whereas Louvain drops off more sharply (about 0.80 and 0.75).

The cluster size distributions reveal the structural consequences of the algorithms, with Louvain producing a relatively healthy, modular distribution. While there are a few larger communities (peaking around 250.000 items), there are dozens of robust, mid-sized clusters (50-150k range). This provides evidence that Louvain finds distinct, heavily propulated sub-networks based on density. 

Spectral's distribution is heavily skewed maintaining the 'gravity well' problem, where one cluster dominates the field, while the other are microscopic in comparison.

## Further K Exploration **Draft**
We are comparing both models at a fixed parameter of $k = 322$, which provides a controlled baseline, Spectral Clustering deomnstrates a structural advantage, particularly in the broader recommendation brackets (maintaining higher Top-5 and Top-10 scores than Louvain). However, forcing Spectral Clustering to match the dynamic community count discovered by Louvain introduces a significant methological constraint. Louvain optimizes for network modularity and density, naturally settling on 322 communities as its ideal structural representation of the data. Spectral Clustering relies on identifying optimal mathematical cuts within the graph's eigenspace. 

By artificially restricting Spectral to 322 partitions, we are likely handicapping its underlying logic and forcing distinct semantic micro-communities to remain merged. This leads us to hypothesize that 322 is not the optimal parameter for Spectral in this high-dimensional space. To evaluate its true predicitve potential, we must decouple its parameterization from Louvain's density-based results, mathematically determine Spectral's own optimal number of clusters, and analyze its unconstrained performance. 

#### The Search of the Optimal K (Silhouettte and Eigengap Heuristics)
In tuning Spectral Clustering, we have had to pick $k$ as a hyperparameter, this $k$ has not been arbitrarily chosen. We started by calculating the *silhouette/eigengap* score for every $k$ up to 322, which was the optimal $k$ found by Louvain, seeing as the score kept increasing the interval was brought up to 1500, with sampling at every 50th $k$, the optimal $k$ was found at 700. To more accurately pin-point the exact $k$ we calculated silhouette scores of surrounding $k$s (675-725) finding $k = 720$ to be optimal. 

While we previously utilized a **Delta WCSS** threshold to determine $k=55$ for representative-based models, graph-based models requires a different form of validation in choosing the optimal hyperparameter $k$. 

- Eigengap Heuristic: We analyze the eigenvalues $\lambda$ of the graph Laplacian to find the largest difference between consecutive values, defined as $\delta_k = \lambda_{k+1} - \lambda_k$. A maximized gap indicates the number of stable, natural communities within the network density. 

- Average Silhouette Coefficient: Because spectral clustering projects the sparse TF-IDF data into a new, low-dimensional coordinate system, we use the silhouette coefficient to evaluate the quality of the resulting clusters within the **spectral embedding space**. 

- Davies-Bouldin Index (DBI): To provide a final check on the 'efficiency' of our high-granularity partition, we utilize the DBI. This metric identifies the average similarity between each cluster $i$ and its most similar neighbor $j$, where $s$ is the cluster diameter and $d_{ij}$ is the distance between centroids:

$$DBI = \frac{1}{k}\sum_{i=1}^kmax_{j \neq i}(\frac{s_i+s_j}{d_{ij}})$$

For any individual playlist $i$, the silhouette scores $s(i)$ is calculated by compaing its average distance to other points in the same cluster, i.e. its cohesion $a(i)$ against its average distance to points in the nearest neighboring cluster, i.e. its separation $b(i)$:

$$s(i) = \frac{b(i) - a(i)}{max(a(i), b(i))}$$

The means of these individual scores across the entire contextual dataset is used in guiding the selection of $k$ because of these following properties:
- A high average silhouette scores (approaching 1) indicated that the spectral transformation has successfully mapped the complex, arbitrary shapes of user-generated musical contexts into well-separated, compact clusters in the eigenvector space. 
- While the eigengap heuristic identifies candidate values for $k$ based on the graph's connectivity, the silhouette coefficient provides a geometric 'sanity check' to ensure these partitions are cohesive. 
- Since human musical curation is often nested or high-dimensional, scores near 0 alert us to overlapping contexts where $k$ may be too high, preventing the 'gravity well' effect previously observed in the K-Means implementation

By calculating this metric within the spectral embedding space, we can verify if the graph-based network representation offers a higher 'semantic purity' than our previous representative-based baseline.

In [ ]:
# Displaying of the silhouette scores, for different k values
path_img_1 = "scripts/data/spectral_k_analysis_spotify_0_322.png"
path_img_2 = "scripts/data/spectral_k_analysis_spotify_0_1500.png"
path_img_3 = "scripts/data/spectral_k_analysis_spotify_OPT720.png"

# 1x3 subplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(plt.imread(path_img_1))
axes[0].set_title("Spectral K Analysis (0-322)")
axes[0].axis('off')
axes[1].imshow(plt.imread(path_img_2))
axes[1].set_title("Spectral K Analysis (0-1500)")
axes[1].axis('off')
axes[2].imshow(plt.imread(path_img_3))
axes[2].set_title("Spectral K Analysis (675-725)")
axes[2].axis('off')
plt.tight_layout()
plt.show()

The optimal $k$ was found to be 720, based on both the silhouette score maximized and a strong supporting DBI score, which was relatively low. They are not in agreement with eigengap which states that the optimal $k$ would be 55.

#### Executing Spectral Clustering - $k = 720$

In [ ]:
# Initialize Spectral Clustering instance, and load the graph from the shared builder
from graph.knn.spectral_clustering import SpectralGraphClustering
Spectral = SpectralGraphClustering(graph=graph_builder.G, n_clusters=720, graph_config_name=graph_config_name)

In [ ]:
# Get the cluster_col name, which we use to accesss / save the data frame
spectral_col = getattr(Spectral, "cluster_col", None)
# Report output directory for Spectral
spectral_report_out = Spectral.report_dir

In [ ]:
# Check if the Spectral column already exists to avoid redundant computation
if spectral_col in df.columns:
    print(f"\n[SKIP] {Spectral.algo_name} already exists in column '{spectral_col}'.")

# If the column doesn't exist, we run the full pipeline for Spectral clustering
else:
    print(f"\n{'='*50}")
    print(f"Executing Pipeline for: {Spectral.algo_name}")
    print(f"{'='*50}")

    # Pipeline execution
    df, target_col = Spectral.run_pipeline(df, unique_texts, tfidf_matrix)
    # Report creation
    Spectral.create_report()

    # If the pipeline produced a new target column, we save the updated dataframe with the new labels
    if target_col and target_col in df.columns:
        from preprocessing.preprocessor import FULLY_PROCESSED_PARQUET
        print(f"[INFO] Saving updated labels to {FULLY_PROCESSED_PARQUET}...")
        df.to_parquet(FULLY_PROCESSED_PARQUET, index=False)

### Evaluation
#### Spectral Clustering Evaluation

In [ ]:
# Safety Net
if spectral_col in df.columns:
    os.makedirs(spectral_report_out, exist_ok=True)
    spectral_report_path = os.path.join(spectral_report_out, f"evaluation_metrics_{spectral_col}.txt")
    # Skip Logic
    if os.path.exists(spectral_report_path):
        print(f"\n[SKIP] Evaluation for {Spectral.algo_name} already exists at '{spectral_report_path}'.")
    else:
        print(f"\n[INFO] Column '{spectral_col}' exists but no evaluation report found. Running eval for {Spectral.algo_name}...")
        eval(df=df,
             cluster_col=spectral_col,
             unique_texts=unique_texts,
             tfidf_matrix=tfidf_matrix,
             sample_frac=0.1,
             output_dir=spectral_report_out)

#### Spectral vs. Louvain

In [ ]:
# Printing the two graphs side by side for comparison
if os.path.exists(spectral_report_path) and os.path.exists(louvain_report_path):
    print("\n[INFO] Displaying evaluation comparison graphs for Spectral and Louvain...")
    
    spectral_f01 = os.path.join(spectral_report_out, "f01_comparison.png")
    louvain_f01 = os.path.join(louvain_report_out, "f01_comparison.png")
    
    if os.path.exists(spectral_f01) and os.path.exists(louvain_f01):
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        axes[0].imshow(plt.imread(spectral_f01))
        axes[0].set_title(f"{Spectral.algo_name} F0.1 Comparison")
        axes[0].axis('off')
        
        axes[1].imshow(plt.imread(louvain_f01))
        axes[1].set_title(f"{Louvain.algo_name} F0.1 Comparison")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("[WARNING] One or both F0.1 comparison graphs are missing. Cannot display side by side.")

The evaluation results for our graph-based algorithms, Spectral Clustering and Louvain Community Detection, demonstrate a drastic improvement over the geometric approached applied in Module 1. Both graph-based methods achieved exceptional predictive performance, with Top-1 Cluster $F_{0.1}$ scores peaking around 0.9 to 0.92 across most test fractions. This is a great improvement compared to the maximum scores of roughly 0.69 for K-means and 0.78 for BIRCH observed previously. 

Louvain relies on greedy local heuristics to iteratively merge nodes that provide a gain in modularity. This
may perhaps constrain its semantic boundary precision due to two main factors. Firstly, its greedy nature makes it
susceptible to converging on local optima, as it only takes steps to improve modularity. Furthermore, it suffers from
a resolution limit, as its objective function evaluates local density against the total edge count (which in our case is
very high), where small communities may be erroneously merged to maximize the global score. On the other hand,
spectral clustering evaluates the structural flow of the entire network simultaneously, hence not being affected by these
limitations.